# Week 3 — Gaussian Processes, SVMs and query selection

Update the observations, assess GP models and SVM diagnostics, and select the third-round queries.


## 1. Load the observations


In [34]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.spatial.distance import cdist
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_val_score
from sklearn.metrics import pairwise_distances

import warnings
from sklearn.exceptions import ConvergenceWarning



working_dir = Path.cwd()
search_folders = [
    working_dir / "Data", working_dir.parent / "Data",
    working_dir.parent / "Week_02" / "Data",
    working_dir.parent.parent / "Week_02" / "Data",
    working_dir.parent / "2. Module 13" / "updated_data_week_2",
    working_dir.parent.parent / "2. Module 13" / "updated_data_week_2",
]
data_folder = next((p for p in search_folders if all((p / f"function_{i}.csv").is_file() for i in range(1, 9))), None)
if data_folder is None:
    raise FileNotFoundError(f"Could not locate function_1.csv through function_8.csv near {working_dir}")
print("Loading observations from:", data_folder)


Loading observations from: C:\Users\charl\OneDrive - thesuperyachtgroup.com\Desktop\Imperial AI course\7. Capstone Project\3. Module 14\Data


In [35]:
data = {}
for i in range(1, 9):
    df = pd.read_csv(data_folder / f"function_{i}.csv")
    df = df.loc[:, ~df.columns.str.match(r"^Unnamed:")]
    df.columns = df.columns.str.strip()
    if "output" not in df.columns:
        raise ValueError(f"function_{i}.csv is missing an output column")
    data[i] = df
print(pd.DataFrame([{"Function": i, "Observations": len(df), "Dimensions": len(df.columns)-1} for i, df in data.items()]).to_string(index=False))

 Function  Observations  Dimensions
        1            12           2
        2            12           2
        3            17           3
        4            32           4
        5            22           4
        6            22           5
        7            32           6
        8            42           8


## 2. Add the latest outputs and assess improvements


In [36]:
round2_points = {
    1: [0.999973, 0.999843],
    2: [0.655286, 0.929896],
    3: [0.397530, 0.090477, 0.447319],
    4: [0.480756, 0.474000, 0.229059, 0.429178],
    5: [0.514842, 0.856105, 0.961506, 0.998492],
    6: [0.246549, 0.344645, 0.479525, 0.733976, 0.025332],
    7: [0.080334, 0.317465, 0.485517, 0.055569, 0.406883, 0.931976],
    8: [0.078127, 0.668711, 0.054188, 0.209488,
        0.863552, 0.717799, 0.074439, 0.866300]
}

round2_outputs = {
    1: 1.868495640628433e-192,
    2: 0.36601049236039945,
    3: -0.06086980027965498,
    4: -3.743172382958963,
    5: 2765.8197868846855,
    6: -0.45851061638506707,
    7: 0.975232656186371,
    8: 9.62112231085
}

expected_week2_counts = {1: 11, 2: 11, 3: 16, 4: 31, 5: 21, 6: 21, 7: 31, 8: 41}
counts = {i: len(data[i]) for i in range(1, 9)}
if counts == {i: n + 1 for i, n in expected_week2_counts.items()}:
    print("Week 3 observations already included; no rows appended.")
elif counts == expected_week2_counts:
    for i in range(1, 9):

        new_row = pd.DataFrame(
            [round2_points[i] + [round2_outputs[i]]],
            columns=data[i].columns
        )

        data[i] = pd.concat(
            [data[i], new_row],
            ignore_index=True
        )

    print("Week 3 observations appended to all eight functions.")
else:
    raise ValueError(f"Unexpected observation counts: {counts}. Check that the CSVs contain Week 2 or Week 3 cumulative observations.")


Week 3 observations already included; no rows appended.


## Compare with previous best results


In [37]:
performance_rows = []
for i in range(1, 9):
    previous_best = data[i].iloc[:-1]["output"].max()
    new_output = data[i].iloc[-1]["output"]
    performance_rows.append({"Function": i, "Previous best": previous_best, "New output": new_output,
                             "Change": new_output - previous_best, "Updated best": data[i]["output"].max(),
                             "New best?": new_output > previous_best})
performance_summary = pd.DataFrame(performance_rows)
print(performance_summary.to_string(index=False, float_format=lambda x: f"{x:.6f}"))

 Function  Previous best  New output     Change  Updated best  New best?
        1       0.000000    0.000000  -0.000000      0.000000      False
        2       0.611205    0.366010  -0.245195      0.611205      False
        3      -0.020167   -0.060870  -0.040703     -0.020167      False
        4      -0.300645   -3.743172  -3.442528     -0.300645      False
        5    2329.015854 2765.819787 436.803933   2765.819787       True
        6      -0.714265   -0.458511   0.255754     -0.458511       True
        7       1.950612    0.975233  -0.975380      1.950612      False
        8       9.821986    9.621122  -0.200864      9.821986      False


## Refit Gaussian Process models


In [38]:
models_week3 = {}
gp_rows = []
for i in range(1, 9):
    df = data[i]
    X = df.drop(columns="output").values
    y = df["output"].values
    kernel = RBF(length_scale=1.0, length_scale_bounds=(1e-5, 1e5))
    gp = GaussianProcessRegressor(kernel=kernel, normalize_y=True, random_state=42, n_restarts_optimizer=10)
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always", ConvergenceWarning)
        gp.fit(X, y)
    models_week3[i] = gp
    gp_rows.append({"Function": i, "Fitted kernel": str(gp.kernel_),
                    "Convergence warnings": len([w for w in caught if issubclass(w.category, ConvergenceWarning)])})
print(pd.DataFrame(gp_rows).to_string(index=False))

 Function            Fitted kernel  Convergence warnings
        1  RBF(length_scale=1e-05)                     1
        2 RBF(length_scale=0.0427)                     0
        3  RBF(length_scale=0.202)                     0
        4  RBF(length_scale=0.495)                     0
        5  RBF(length_scale=0.413)                     0
        6  RBF(length_scale=1e-05)                     1
        7  RBF(length_scale=0.244)                     0
        8  RBF(length_scale=0.931)                     0


## Check predictions and uncertainty


In [39]:
rng = np.random.default_rng(42)
diagnostic_candidates = {}
diagnostic_rows = []
for i in [1, 6]:
    df = data[i]
    gp = models_week3[i]
    X_candidates = rng.uniform(0, 1, size=(100000, len(df.columns)-1))
    diagnostic_candidates[i] = X_candidates
    mean, std = gp.predict(X_candidates, return_std=True)
    diagnostic_rows.append({"Function": i, "Mean min": mean.min(), "Mean max": mean.max(),
                            "Mean range": np.ptp(mean), "Std min": std.min(),
                            "Std max": std.max(), "Std range": np.ptp(std)})
print(pd.DataFrame(diagnostic_rows).to_string(index=False, float_format=lambda x: f"{x:.10f}"))

 Function      Mean min      Mean max   Mean range      Std min      Std max    Std range
        1 -0.0003005052 -0.0000001770 0.0003003282 0.0000342068 0.0009966631 0.0009624562
        6 -1.5012665979 -1.5012665979 0.0000000000 0.5421470063 0.5421470063 0.0000000000


## Create high- and low-performing SVM classes


In [40]:
svm_data = {}
class_rows = []
for i in range(1, 9):
    df = data[i]
    X = df.drop(columns="output").values
    y = df["output"].values
    threshold = np.median(y)
    y_class = (y > threshold).astype(int)
    svm_data[i] = {"X": X, "y_class": y_class, "threshold": threshold}
    class_rows.append({"Function": i, "Median threshold": threshold,
                       "Low": (y_class == 0).sum(), "High": (y_class == 1).sum()})
print(pd.DataFrame(class_rows).to_string(index=False, float_format=lambda x: f"{x:.6f}"))

 Function  Median threshold  Low  High
        1          0.000000    6     6
        2          0.229792    6     6
        3         -0.094190    9     8
        4        -15.756741   16    16
        5         64.431793   11    11
        6         -1.446370   11    11
        7          0.088196   16    16
        8          7.940817   21    21


## Compare linear and RBF SVMs


In [41]:
svm_rows = []
for i in range(1, 9):
    X, y_class = svm_data[i]["X"], svm_data[i]["y_class"]
    linear_svm = make_pipeline(StandardScaler(), SVC(kernel="linear"))
    rbf_svm = make_pipeline(StandardScaler(), SVC(kernel="rbf"))
    linear_scores = cross_val_score(linear_svm, X, y_class, cv=3, scoring="accuracy")
    rbf_scores = cross_val_score(rbf_svm, X, y_class, cv=3, scoring="accuracy")
    svm_rows.append({"Function": i, "Linear accuracy": linear_scores.mean(), "RBF accuracy": rbf_scores.mean()})
svm_summary = pd.DataFrame(svm_rows)
print(svm_summary.to_string(index=False, float_format=lambda x: f"{x:.3f}"))

 Function  Linear accuracy  RBF accuracy
        1            0.167         0.500
        2            0.667         0.833
        3            0.344         0.522
        4            0.627         0.782
        5            0.649         0.458
        6            0.821         0.768
        7            0.785         0.815
        8            0.857         0.833


## Reassess exploration and exploitation


In [42]:
strategy_summary = pd.DataFrame({
    "Function": range(1, 9),

    "Latest_Query_Improved": [
        *[bool(x) for x in performance_summary["New best?"]]
    ],

    "GP_Usable": [
        False, True, True, True,
        True, False, True, True
    ],

    "Linear_SVM_Accuracy": svm_summary["Linear accuracy"].tolist(),
    "RBF_SVM_Accuracy": svm_summary["RBF accuracy"].tolist(),

    "Proposed_Strategy": [
        "Exploration",
        "Balanced UCB",
        "Balanced UCB",
        "UCB / moderate exploitation",
        "Greater exploitation",
        "SVM-informed / exploration",
        "Balanced UCB",
        "Balanced UCB"
    ]
})

print(strategy_summary.to_string(index=False))


 Function  Latest_Query_Improved  GP_Usable  Linear_SVM_Accuracy  RBF_SVM_Accuracy           Proposed_Strategy
        1                  False      False             0.166667          0.500000                 Exploration
        2                  False       True             0.666667          0.833333                Balanced UCB
        3                  False       True             0.344444          0.522222                Balanced UCB
        4                  False       True             0.627273          0.781818 UCB / moderate exploitation
        5                   True       True             0.648810          0.458333        Greater exploitation
        6                   True      False             0.821429          0.767857  SVM-informed / exploration
        7                  False       True             0.784848          0.815152                Balanced UCB
        8                  False       True             0.857143          0.833333                Balanced UCB


## Select GP + UCB candidates


In [43]:
kappa = 2.5
n_candidates = 100000
ucb_functions = [2, 3, 4, 5, 7, 8]
ucb_selected = {}
ucb_rows = []
rng = np.random.default_rng(42)
for i in ucb_functions:
    df = data[i]
    gp = models_week3[i]
    X_candidates = rng.uniform(0, 1, size=(n_candidates, len(df.columns)-1))
    mean, std = gp.predict(X_candidates, return_std=True)
    ucb = mean + kappa * std
    best_index = np.argmax(ucb)
    selected_point = X_candidates[best_index]
    ucb_selected[i] = selected_point
    ucb_rows.append({"Function": i, "Predicted mean": mean[best_index],
                     "Uncertainty": std[best_index], "UCB": ucb[best_index],
                     "Coordinates": " ".join(f"{x:.6f}" for x in selected_point)})
print(pd.DataFrame(ucb_rows).to_string(index=False, float_format=lambda x: f"{x:.6f}"))

 Function  Predicted mean  Uncertainty         UCB                                                             Coordinates
        2        0.449335     0.185666    0.913500                                                       0.709627 0.881953
        3       -0.076907     0.077556    0.116983                                              0.732035 0.997300 0.015653
        4       -0.064270     0.420976    0.988170                                     0.391976 0.424757 0.494823 0.464424
        5     2631.682259   240.331386 3232.510725                                     0.671629 0.981118 0.990458 0.985189
        7        1.662117     0.258204    2.307626                   0.021549 0.313334 0.360216 0.141732 0.295328 0.686548
        8        9.635913     0.434286   10.721628 0.025529 0.451238 0.098209 0.031391 0.412421 0.138573 0.114934 0.994361


## Compare candidates with current best observations


In [44]:
distance_rows = []
for i in ucb_functions:
    df = data[i]
    current_best_point = df.drop(columns="output").loc[df["output"].idxmax()].values.astype(float)
    proposed_point = ucb_selected[i]
    distance_rows.append({"Function": i, "Current best output": df["output"].max(),
                          "Distance from best": np.linalg.norm(proposed_point-current_best_point)})
print(pd.DataFrame(distance_rows).to_string(index=False, float_format=lambda x: f"{x:.6f}"))

 Function  Current best output  Distance from best
        2             0.611205            0.045156
        3            -0.020167            0.846074
        4            -0.300645            0.137608
        5          2765.819787            0.203041
        7             1.950612            0.161151
        8             9.821986            0.839162


## Function 1 — maximum-distance exploration


In [45]:
from sklearn.metrics import pairwise_distances


i = 1

X_existing = data[i].drop(columns="output").values

rng = np.random.default_rng(42)

X_candidates = rng.uniform(
    0,
    1,
    size=(100000, X_existing.shape[1])
)

distances = pairwise_distances(
    X_candidates,
    X_existing
)

nearest_distances = distances.min(axis=1)

best_index = np.argmax(nearest_distances)

f1_selected = X_candidates[best_index]

print(f"Function 1 max-distance point: {np.array2string(f1_selected, precision=8)}")
print(f"Distance to nearest observation: {nearest_distances[best_index]:.6f}")

Function 1 max-distance point: [4.50627487e-05 3.06208459e-03]
Distance to nearest observation: 0.321671


## Function 6 — SVM-informed selection


In [46]:
i = 6

X = svm_data[i]["X"]
y_class = svm_data[i]["y_class"]

f6_svm = make_pipeline(
    StandardScaler(),
    SVC(kernel="linear")
)

f6_svm.fit(X, y_class)

rng = np.random.default_rng(42)

X_candidates = rng.uniform(
    0,
    1,
    size=(100000, X.shape[1])
)

decision_scores = f6_svm.decision_function(X_candidates)

best_index = np.argmax(decision_scores)

f6_svm_candidate = X_candidates[best_index]

best_obs_index = data[6]["output"].idxmax()
current_best = data[6].drop(columns="output").loc[best_obs_index].values.astype(float)
distance = np.linalg.norm(f6_svm_candidate - current_best)
print(f"Function 6 linear SVM point: {np.array2string(f6_svm_candidate, precision=8)}")
print(f"Decision score: {decision_scores[best_index]:.6f}; distance from current best: {distance:.6f}")

Function 6 linear SVM point: [0.01375812 0.14792993 0.44559686 0.99484501 0.01346948]
Decision score: 5.272052; distance from current best: 0.402781


## Check Function 6 candidate


In [47]:
i = 6

X = svm_data[i]["X"]
y_class = svm_data[i]["y_class"]

X_high = X[y_class == 1]

high_distances = np.linalg.norm(
    X_high - f6_svm_candidate,
    axis=1
)

print(f"Function 6: nearest high-performing observation = {high_distances.min():.6f}; "
      f"mean distance = {high_distances.mean():.6f}; high-performing observations = {len(X_high)}")

Function 6: nearest high-performing observation = 0.402781; mean distance = 0.886477; high-performing observations = 11


## Validate the final queries


In [48]:
final_points = {
    1: f1_selected,
    2: ucb_selected[2],
    3: ucb_selected[3],
    4: ucb_selected[4],
    5: ucb_selected[5],
    6: f6_svm_candidate,
    7: ucb_selected[7],
    8: ucb_selected[8]
}

selection_methods = {
    1: "Maximum-distance exploration",
    2: "RBF GP + UCB",
    3: "RBF GP + UCB",
    4: "RBF GP + UCB",
    5: "RBF GP + UCB",
    6: "Linear SVM",
    7: "RBF GP + UCB",
    8: "RBF GP + UCB"
}

validation_rows = []
for i in range(1, 9):
    point = np.asarray(final_points[i])
    X_existing = data[i].drop(columns="output").values
    in_bounds = bool(np.all((point >= 0) & (point <= 1)))
    distances = np.linalg.norm(X_existing - point, axis=1)
    duplicate = bool(np.any(np.all(np.isclose(X_existing, point), axis=1)))
    validation_rows.append({"Function": i, "Method": selection_methods[i], "In bounds": in_bounds,
                            "Duplicate": duplicate, "Nearest distance": distances.min(),
                            "Max-distance correct": bool(np.allclose(point, f1_selected)) if i == 1 else "—"})
validation_summary = pd.DataFrame(validation_rows)
print(validation_summary.to_string(index=False, float_format=lambda x: f"{x:.6f}"))
assert validation_summary["In bounds"].all() and not validation_summary["Duplicate"].any()
assert validation_summary.loc[0, "Max-distance correct"] is True or validation_summary.loc[0, "Max-distance correct"] == True

 Function                       Method  In bounds  Duplicate  Nearest distance Max-distance correct
        1 Maximum-distance exploration       True      False          0.321671                 True
        2                 RBF GP + UCB       True      False          0.045156                    —
        3                 RBF GP + UCB       True      False          0.306635                    —
        4                 RBF GP + UCB       True      False          0.137608                    —
        5                 RBF GP + UCB       True      False          0.203041                    —
        6                   Linear SVM       True      False          0.402781                    —
        7                 RBF GP + UCB       True      False          0.161151                    —
        8                 RBF GP + UCB       True      False          0.800486                    —


## Format points for submission


In [49]:
for i in range(1, 9):
    submission = "-".join(
        f"{value:.6f}"
        for value in final_points[i]
    )

    print(f"Function {i}: {submission}")


Function 1: 0.000045-0.003062
Function 2: 0.709627-0.881953
Function 3: 0.732035-0.997300-0.015653
Function 4: 0.391976-0.424757-0.494823-0.464424
Function 5: 0.671629-0.981118-0.990458-0.985189
Function 6: 0.013758-0.147930-0.445597-0.994845-0.013469
Function 7: 0.021549-0.313334-0.360216-0.141732-0.295328-0.686548
Function 8: 0.025529-0.451238-0.098209-0.031391-0.412421-0.138573-0.114934-0.994361
